# 两个137.055 MHz事件的Starlink目标确定

把“两个事件为什么属于Starlink VHF、为什么最可能对应STARLINK-32838”拆成可逐格检查的证据链。
证据分成三层：

1. 两个事件是否是真实、CRC有效、符合已知Starlink VHF profile的LoRa包；
2. Event 1中推断的六维状态向量在全部活动卫星TLE中最匹配哪颗目标；
3. Event 2能否通过同ID、连续序号、100 s间隔和差分多普勒与Event 1关联到同一目标。

**重要限制：payload中用于状态向量的24字节在公开格式资料中仍标为TBD。ECEF int32/1e5是经过物理合理性和TLE外部匹配支持的反向解释，不是SpaceX官方公开字段定义。**


## 0. 结论边界

- Event 1可以通过推断状态向量进行全目录独立盲筛。
- Event 2的对应状态块为零，不能独立完成同样的卫星目录识别。
- Event 2只能继承Event 1的候选身份，再用包关系和差分多普勒检验是否来自同一发射体。
- protocol/spacecraft-like ID 5927不是NORAD编号，不能直接用它查询STARLINK-32838。
- TLE最佳模型时刻只是补偿旧TLE沿轨误差的拟合量，不是实际接收时刻。


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import binascii
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from skyfield.api import load
from skyfield.framelib import itrs
from skyfield.iokit import parse_tle_file

# 本Notebook所有论文图统一使用Matplotlib默认的DejaVu Sans。
PLOT_FONT_FAMILY = 'DejaVu Sans'
PLOT_FONT_WEIGHT = 400
plt.rcParams.update({
    'font.family': PLOT_FONT_FAMILY,
    'font.weight': PLOT_FONT_WEIGHT,
    'axes.labelweight': PLOT_FONT_WEIGHT,
    'axes.titleweight': PLOT_FONT_WEIGHT,
    'mathtext.fontset': 'dejavusans',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda value: f'{value:.6f}')


## 1. 输入、参数和输出目录

默认读取file0和file12的LFdemod结果，以及观测当天北京时间上午9:30-10:00下载的Active Satellites TLE快照。主盲筛固定让全部唯一NORAD目标参与精确时间优化。


In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'project_config.py').is_file():
            return candidate
    raise FileNotFoundError('Run from this repository or one of its subdirectories')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_config import DATA_DIR, RAW_DATA_DIR, RESULTS_DIR, file_start_utc
ROOT = PROJECT_ROOT
EVENT_DIRS = {
    'Event 1': RESULTS_DIR / '2_demodulation' / 'file0',
    'Event 2': RESULTS_DIR / '2_demodulation' / 'file12',
}
TLE_PATH = DATA_DIR / 'orbital' / 'Active_2025-04-15.tle'
OUTPUT_DIR = RESULTS_DIR / '3_satellite_identification'

TIME_BOUND_S = 180.0
VELOCITY_WEIGHT_S = 100.0
SAVE_RESULTS = True

RAW_SAMPLE_RATE_HZ = 480e6
RF_FREQUENCY_HZ = 137_055_000.0
LIGHT_SPEED_KM_S = 299_792.458
GPS_EPOCH_UTC = datetime(1980, 1, 6, tzinfo=timezone.utc)
GPS_MINUS_UTC_S = 18.0
STATION_CENTER_PATH = RESULTS_DIR / '2_waterfall' / 'e08_station_center_wgs84.csv'
station_center = pd.read_csv(STATION_CENTER_PATH)
if len(station_center) != 1 or str(station_center.iloc[0]['station']).upper() != 'E08':
    raise ValueError('站心文件应只包含一行E08坐标')
station_center = station_center.iloc[0]
SITE_LATITUDE_DEG = float(station_center['latitude_deg_north'])
SITE_LONGITUDE_DEG = float(station_center['longitude_deg_east'])
SITE_ELEVATION_M = float(station_center['ellipsoidal_height_m'])

for event_dir in EVENT_DIRS.values():
    assert (event_dir / 'summary.json').is_file()
    assert (event_dir / 'packet_000_frame.bin').is_file()
assert TLE_PATH.is_file()
assert STATION_CENTER_PATH.is_file()

print('TLE:', TLE_PATH)
print('Output:', OUTPUT_DIR)
print(
    'Station center: E08',
    f'lat={SITE_LATITUDE_DEG:.9f} deg,',
    f'lon={SITE_LONGITUDE_DEG:.9f} deg,',
    f'h={SITE_ELEVATION_M:.3f} m',
)
print('Exact-search scope: all unique NORAD targets in the active-satellite catalogue')


## 2. 读取两个解调帧并独立复算CRC

LFdemod输出的frame由81-byte payload和2-byte payload CRC组成，共83 bytes。这里不直接相信summary中的crc_valid，而是从二进制帧重新计算CRC。CRC有效只能说明解调出的bit序列内部自洽；它和频率、LoRa参数及固定字段共同构成Starlink VHF profile证据。

In [ ]:
def lora_payload_crc(payload):
    sequence = binascii.crc_hqx(payload[:-2], 0)
    low = ((sequence & 0xFF) ^ payload[-1]) & 0xFF
    high = (((sequence >> 8) ^ payload[-2]) & 0xFF)
    return bytes((low, high))

summaries = {}
frames = {}
payloads = {}
frame_rows = []

for label, event_dir in EVENT_DIRS.items():
    summary = json.loads((event_dir / 'summary.json').read_text(encoding='utf-8'))
    frame = (event_dir / 'packet_000_frame.bin').read_bytes()
    assert len(frame) == 83, (label, len(frame))
    payload, received_crc = frame[:81], frame[81:]
    calculated_crc = lora_payload_crc(payload)
    packet = summary['packets'][0]

    summaries[label] = summary
    frames[label] = frame
    payloads[label] = payload
    frame_rows.append({
        'event': label,
        'raw_file': summary['input_path'],
        'frame_sha256': hashlib.sha256(frame).hexdigest(),
        'frame_bytes': len(frame),
        'payload_bytes': len(payload),
        'header_valid': bool(packet['header_valid']),
        'coding_rate': packet['coding_rate_label'],
        'received_crc': received_crc.hex(),
        'calculated_crc': calculated_crc.hex(),
        'crc_valid_independent': calculated_crc == received_crc,
        'starlink_profile_match': bool(packet['starlink_profile_match']),
        'carrier_offset_hz': float(packet['carrier_offset_hz']),
        'observed_center_hz': float(packet['observed_center_hz']),
    })

frame_table = pd.DataFrame(frame_rows).set_index('event')
frame_table


## 3. 检查Starlink VHF已知profile和两个包之间的关系

当前公开反向工程资料和TinyGS配置支持一组固定字段与LoRa参数。这里把判断拆开显示，不把“频率一致”或“单个固定字节一致”当作唯一证据。


In [ ]:
profile_rows = []
relationship_rows = []

for label, payload in payloads.items():
    packet = summaries[label]['packets'][0]
    lora = summaries[label]['config']['lora']
    profile_rows.append({
        'event': label,
        'rf_center_is_137p055_mhz': summaries[label]['config']['rf_frequency_hz'] == 137_055_000,
        'bw_hz': lora['bandwidth_hz'],
        'sf': lora['spreading_factor'],
        'preamble_symbols': lora['preamble_symbols'],
        'sync_word_hex': hex(lora['sync_word']),
        'explicit_header': packet['has_explicit_header'],
        'payload_81_bytes': len(payload) == 81,
        'packet_type_0xcc': payload[5] == 0xCC,
        'bytes_6_7_zero': payload[6:8] == bytes(2),
        'source_0xd0': payload[8] == 0xD0,
        'remaining_length_0x45': payload[11] == 0x45,
        'frame_format_0x05': payload[12] == 0x05,
        'crc_valid': frame_table.loc[label, 'crc_valid_independent'],
    })
    relationship_rows.append({
        'event': label,
        'message_number_u24': int.from_bytes(payload[0:3], 'little'),
        'protocol_id_u16': int.from_bytes(payload[3:5], 'little'),
        'unix_candidate': int.from_bytes(payload[14:18], 'little'),
        'unix_candidate_utc': datetime.fromtimestamp(
            int.from_bytes(payload[14:18], 'little'), tz=timezone.utc
        ).isoformat(),
        'state_block_all_zero': payload[38:62] == bytes(24),
    })

profile_table = pd.DataFrame(profile_rows).set_index('event')
relationship_table = pd.DataFrame(relationship_rows).set_index('event')
display(profile_table)
display(relationship_table)


两个包的直接关系应满足：同一protocol ID、消息序号加1、Unix-like时间候选加100 s、相同物理层和frame profile。下一格用原始文件序号和文件内样本时刻计算实际preamble间隔。


In [ ]:
# 绝对preamble UTC由各DAT文件的acquisition metadata文件起点后重建。
# 两事件间隔也统一使用这两个acquisition-metadata-derived时刻之差，不再使用文件序号假定连续采样得到的99.9991 s。
metadata_preamble_utc = {}
for label, summary in summaries.items():
    packet = summary['packets'][0]
    raw_path = Path(summary['input_path'])
    start_utc = file_start_utc(raw_path.name).to_pydatetime()
    metadata_preamble_utc[label] = start_utc + timedelta(
        seconds=float(packet['preamble_start_in_file_seconds'])
    )

message_increment = (
    relationship_table.loc['Event 2', 'message_number_u24']
    - relationship_table.loc['Event 1', 'message_number_u24']
)
unix_increment_s = (
    relationship_table.loc['Event 2', 'unix_candidate']
    - relationship_table.loc['Event 1', 'unix_candidate']
)
measured_spacing_s = (
    metadata_preamble_utc['Event 2'] - metadata_preamble_utc['Event 1']
).total_seconds()

packet_pair_summary = pd.Series({
    'same_protocol_id': (
        relationship_table.loc['Event 1', 'protocol_id_u16']
        == relationship_table.loc['Event 2', 'protocol_id_u16']
    ),
    'protocol_id': relationship_table.loc['Event 1', 'protocol_id_u16'],
    'message_increment': message_increment,
    'unix_candidate_increment_s': unix_increment_s,
    'event1_preamble_utc': metadata_preamble_utc['Event 1'].isoformat(),
    'event2_preamble_utc': metadata_preamble_utc['Event 2'].isoformat(),
    'measured_preamble_spacing_s': measured_spacing_s,
    'preamble_spacing_basis': 'difference of acquisition-metadata-derived preamble UTCs',
    'event2_state_block_all_zero': relationship_table.loc['Event 2', 'state_block_all_zero'],
})
packet_pair_summary.to_frame('value')


## 4. Event 1状态时间、ECEF位置和速度

Event 1 payload的1-based bytes 39–62按六个little-endian signed int32、比例因子1e-5解释。前三个数是ECEF位置km，后三个数是ECEF速度km/s。首先只检验它是否形成物理合理的LEO状态，再交给TLE全目录竞争。下面同时输出完整24-byte状态块，以及X、Y、Z、Vx、Vy、Vz各自对应的4个原始字节，便于逐字节复核。



In [ ]:
event1_payload = payloads['Event 1']
unix_seconds = int.from_bytes(event1_payload[14:18], 'little')
gps_week = int.from_bytes(event1_payload[32:34], 'little')
gps_centiseconds = int.from_bytes(event1_payload[34:38], 'little')
gps_state_utc = GPS_EPOCH_UTC + timedelta(
    weeks=gps_week,
    seconds=gps_centiseconds / 100.0 - GPS_MINUS_UTC_S,
)

packet_position_km = np.frombuffer(event1_payload[38:50], dtype='<i4').astype(float) / 1e5
packet_velocity_km_s = np.frombuffer(event1_payload[50:62], dtype='<i4').astype(float) / 1e5

# 保留payload中未经解释的原始状态字节，逐分量展示解码过程。
state_block = event1_payload[38:62]
state_names = ['X', 'Y', 'Z', 'Vx', 'Vy', 'Vz']
state_units = ['km', 'km', 'km', 'km/s', 'km/s', 'km/s']
state_byte_rows = []
for index, (name, unit) in enumerate(zip(state_names, state_units)):
    component_bytes = state_block[4 * index:4 * (index + 1)]
    raw_int32 = int.from_bytes(component_bytes, byteorder='little', signed=True)
    first_byte = 39 + 4 * index
    state_byte_rows.append({
        'component': name,
        'payload_bytes_1based': f'{first_byte}-{first_byte + 3}',
        'raw_bytes_hex': ' '.join(f'{byte:02x}' for byte in component_bytes),
        'raw_bytes_uint8': list(component_bytes),
        'signed_int32_le': raw_int32,
        'scale_factor': 1e-5,
        'decoded_value': raw_int32 * 1e-5,
        'unit': unit,
    })
state_bytes_table = pd.DataFrame(state_byte_rows).set_index('component')

packet_state = pd.Series({
    'unix_candidate_utc': datetime.fromtimestamp(unix_seconds, tz=timezone.utc).isoformat(),
    'gps_state_utc': gps_state_utc.isoformat(),
    'x_ecef_km': packet_position_km[0],
    'y_ecef_km': packet_position_km[1],
    'z_ecef_km': packet_position_km[2],
    'vx_ecef_km_s': packet_velocity_km_s[0],
    'vy_ecef_km_s': packet_velocity_km_s[1],
    'vz_ecef_km_s': packet_velocity_km_s[2],
    'position_norm_km': np.linalg.norm(packet_position_km),
    'velocity_norm_km_s': np.linalg.norm(packet_velocity_km_s),
    'position_is_physical_leo': 6600 <= np.linalg.norm(packet_position_km) <= 7100,
    'velocity_is_physical_leo': 7.0 <= np.linalg.norm(packet_velocity_km_s) <= 8.2,
})

print('Event 1 payload bytes 39-62: 24-byte XYZ/Vxyz state block')
print('hex:', state_block.hex(' '))
display(state_bytes_table)
display(packet_state.to_frame('value'))


## 5. 读取活动卫星TLE并为每颗目标保留最接近事件时刻的记录

盲筛不能先限定为Starlink，也不能只比较人工挑出的两颗候选。目录中每颗目标应独立传播。若一个NORAD ID在文件中出现多条TLE，保留epoch最接近Event 1 GPS状态时刻的一条。


In [ ]:
ts = load.timescale()
state_time = ts.from_datetime(gps_state_utc)
with load.open(str(TLE_PATH)) as tle_file:
    satellites_all = list(parse_tle_file(tle_file, ts))

satellite_by_norad = {}
for satellite in satellites_all:
    norad = int(satellite.model.satnum)
    current = satellite_by_norad.get(norad)
    if current is None:
        satellite_by_norad[norad] = satellite
        continue
    current_delta = abs((current.epoch.utc_datetime() - gps_state_utc).total_seconds())
    new_delta = abs((satellite.epoch.utc_datetime() - gps_state_utc).total_seconds())
    if new_delta < current_delta:
        satellite_by_norad[norad] = satellite

satellites = list(satellite_by_norad.values())
epochs = [sat.epoch.utc_datetime() for sat in satellites]
catalogue_summary = pd.Series({
    'tle_records': len(satellites_all),
    'unique_norad_ids_used': len(satellites),
    'earliest_epoch_utc': min(epochs).isoformat(),
    'latest_epoch_utc': max(epochs).isoformat(),
    'event_gps_state_utc': gps_state_utc.isoformat(),
})
catalogue_summary.to_frame('value')


## 6. 全目录快速盲筛与精确最佳时刻

本节把Event 1 payload推断出的ECEF位置 $\mathbf r_{\rm packet}$ 和速度 $\mathbf v_{\rm packet}$，与活动卫星TLE目录中的**每一个唯一NORAD目标**进行独立比较。主盲筛不会根据快速排名淘汰卫星；快速阶段和精确阶段都从同一完整卫星目录出发。

### 6.1 快速线性时间校正

首先把每颗卫星的TLE传播到payload给出的GPS状态时刻 $t_0$，得到ECEF位置 $\mathbf r_{\rm TLE}(t_0)$ 和速度 $\mathbf v_{\rm TLE}(t_0)$。将位置残差投影到卫星速度方向，得到线性沿轨时间偏移估计：

$$
\Delta t_{\rm linear}=\frac{[\mathbf r_{\rm packet}-\mathbf r_{\rm TLE}(t_0)]\cdot\mathbf v_{\rm TLE}(t_0)}{\mathbf v_{\rm TLE}(t_0)\cdot\mathbf v_{\rm TLE}(t_0)}.
$$

该偏移被限制在 $[-\mathrm{TIME\_BOUND\_S},+\mathrm{TIME\_BOUND\_S}]$，当前即 $[-180,+180]$ s。随后把TLE传播到 $t_0+\Delta t_{\rm linear}$，计算快速位置误差、速度误差和联合评分。这个快速结果用于诊断和给出全目录的初始排名，但**不再用于筛除目标**。

### 6.2 全目录精确时间搜索

对每一颗活动卫星，定义一个独立的时间偏移 $\Delta t$，并在

$$-180\ {\rm s}\leq\Delta t\leq+180\ {\rm s}$$

范围内反复调用Skyfield/SGP4，把该卫星的TLE传播到试探时刻 $t_0+\Delta t$。每个试探时刻计算：

$$
\Delta r(\Delta t)=\left\|\mathbf r_{\rm TLE}(t_0+\Delta t)-\mathbf r_{\rm packet}\right\|,
$$

$$
\Delta v(\Delta t)=\left\|\mathbf v_{\rm TLE}(t_0+\Delta t)-\mathbf v_{\rm packet}\right\|.
$$

精确搜索最小化同一个位置—速度联合评分：

$$
J(\Delta t)=\Delta r(\Delta t)^2+[W\,\Delta v(\Delta t)]^2,
$$

其中默认 $W=100$ s，只是把速度误差换算到等效距离尺度的分析权重，并不是协议常数。代码使用 `scipy.optimize.minimize_scalar(method='bounded')` 在有界区间内迭代寻找 $J$ 的最低点，而不是按照固定时间步长穷举全部时刻。优化器通过有界的一维插值搜索逐渐缩小最低值所在区间；`xatol=1e-4`只是数值停止容差，不代表TLE或事件时间具有0.1 ms的物理精度。

每颗卫星最终保存各自的 `best_offset_s`、`best_time_utc`、最小位置误差、最小速度误差和最小联合评分。全部唯一NORAD目标完成同样优化后，才按照 `joint_score` 生成最终全目录排名，并用数量断言确认没有目标被遗漏。


这里的时间滑动只用于补偿旧TLE可能存在的沿轨相位误差。`best_time_utc = t_0 + best_offset_s`表示**TLE模型与payload六维状态最匹配的模型时刻**，不能解释为新的信号接收UTC，也不表示数据包真实提前或推迟发射。时间滑动可以调整卫星沿轨位置，却不能任意改变轨道平面、速度方向或轨道形状；因此最终必须同时报告位置和速度误差，防止一颗仅经过相似位置但运动方向不同的目标胜出。


In [ ]:
def satellite_state_ecef(satellite, skyfield_time):
    state = satellite.at(skyfield_time)
    xyz, velocity = state.frame_xyz_and_velocity(itrs)
    return np.asarray(xyz.km), np.asarray(velocity.km_per_s)

def score_from_errors(position_error_km, velocity_error_km_s, weight_s=VELOCITY_WEIGHT_S):
    return position_error_km**2 + (weight_s * velocity_error_km_s)**2

def quick_match(satellite):
    position, velocity = satellite_state_ecef(satellite, state_time)
    residual = packet_position_km - position
    offset_s = np.clip(
        np.dot(residual, velocity) / max(np.dot(velocity, velocity), 1e-12),
        -TIME_BOUND_S, TIME_BOUND_S,
    )
    shifted = ts.from_datetime(gps_state_utc + timedelta(seconds=float(offset_s)))
    shifted_position, shifted_velocity = satellite_state_ecef(satellite, shifted)
    position_error = float(np.linalg.norm(shifted_position - packet_position_km))
    velocity_error = float(np.linalg.norm(shifted_velocity - packet_velocity_km_s))
    return {
        'name': satellite.name or '',
        'norad_id': int(satellite.model.satnum),
        'tle_epoch_utc': satellite.epoch.utc_datetime().isoformat(),
        'tle_age_hours': (gps_state_utc - satellite.epoch.utc_datetime()).total_seconds() / 3600,
        'linear_offset_s': float(offset_s),
        'linear_position_error_km': position_error,
        'linear_velocity_error_km_s': velocity_error,
        'quick_score': score_from_errors(position_error, velocity_error),
    }

def precise_match(satellite, weight_s=VELOCITY_WEIGHT_S):
    def objective(offset_s):
        trial = ts.from_datetime(gps_state_utc + timedelta(seconds=float(offset_s)))
        position, velocity = satellite_state_ecef(satellite, trial)
        return score_from_errors(
            np.linalg.norm(position - packet_position_km),
            np.linalg.norm(velocity - packet_velocity_km_s),
            weight_s,
        )

    optimum = minimize_scalar(
        objective,
        bounds=(-TIME_BOUND_S, TIME_BOUND_S),
        method='bounded',
        options={'xatol': 1e-4},
    )
    offset_s = float(optimum.x)
    best_datetime = gps_state_utc + timedelta(seconds=offset_s)
    position, velocity = satellite_state_ecef(
        satellite, ts.from_datetime(best_datetime)
    )
    return {
        'best_offset_s': offset_s,
        'best_time_utc': best_datetime.isoformat(),
        'best_position_error_km': float(np.linalg.norm(position - packet_position_km)),
        'best_velocity_error_km_s': float(np.linalg.norm(velocity - packet_velocity_km_s)),
        'joint_score': float(optimum.fun),
    }

print('已定义传播、快速初筛和精确时间优化函数。')


In [ ]:
quick_rank = pd.DataFrame([quick_match(sat) for sat in satellites])
quick_rank = quick_rank.sort_values('quick_score').reset_index(drop=True)
print('完成快速盲筛目标数:', len(quick_rank))
quick_rank.head(15)


In [ ]:
# 论文主盲筛：活动卫星目录中的每个唯一NORAD目标都参与精确优化。
refine_norads = set(quick_rank['norad_id'].astype(int))

precise_rows = []
for number, norad in enumerate(sorted(refine_norads), start=1):
    satellite = satellite_by_norad[int(norad)]
    row = quick_rank.loc[quick_rank['norad_id'] == int(norad)].iloc[0].to_dict()
    row.update(precise_match(satellite))
    precise_rows.append(row)
    if number % 1000 == 0:
        print(f'已精确优化 {number}/{len(refine_norads)}')

final_rank = pd.DataFrame(precise_rows).sort_values('joint_score').reset_index(drop=True)
assert len(final_rank) == len(satellites), '主盲筛未覆盖全部唯一NORAD目标'
final_rank.index = final_rank.index + 1
final_rank.index.name = 'rank'

winner = final_rank.iloc[0]
runner_up = final_rank.iloc[1]
result_columns = [
    'name', 'norad_id', 'tle_epoch_utc', 'tle_age_hours',
    'best_offset_s', 'best_time_utc', 'best_position_error_km',
    'best_velocity_error_km_s', 'joint_score',
]
display(final_rank[result_columns].head(15))
print('第一名:', winner['name'], int(winner['norad_id']))
print('第一名位置误差 km:', winner['best_position_error_km'])
print('第一名速度误差 m/s:', winner['best_velocity_error_km_s'] * 1000)
print('第二名/第一名评分比:', runner_up['joint_score'] / winner['joint_score'])
print('精确优化目标数/目录唯一NORAD数:', len(final_rank), '/', len(satellites))


## 7. 检查冠军的六个状态分量

只有位置误差小不够。下面分别比较X、Y、Z、Vx、Vy、Vz，并给出向量夹角和相对误差。STARLINK-32838的优势应在位置和速度两方面同时出现。



In [ ]:
winner_norad = int(winner['norad_id'])
winner_satellite = satellite_by_norad[winner_norad]
winner_time = datetime.fromisoformat(str(winner['best_time_utc']))
winner_position_km, winner_velocity_km_s = satellite_state_ecef(
    winner_satellite, ts.from_datetime(winner_time)
)

def vector_angle_deg(first, second):
    cosine = np.dot(first, second) / (np.linalg.norm(first) * np.linalg.norm(second))
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))

position_components = pd.DataFrame({
    'component': ['X', 'Y', 'Z'],
    'packet_km': packet_position_km,
    'tle_km': winner_position_km,
    'packet_minus_tle_km': packet_position_km - winner_position_km,
    'absolute_relative_difference_percent': (
        np.abs(packet_position_km - winner_position_km)
        / np.abs(winner_position_km) * 100
    ),
})
velocity_components = pd.DataFrame({
    'component': ['Vx', 'Vy', 'Vz'],
    'packet_km_s': packet_velocity_km_s,
    'tle_km_s': winner_velocity_km_s,
    'packet_minus_tle_m_s': (packet_velocity_km_s - winner_velocity_km_s) * 1000,
    'absolute_relative_difference_percent': (
        np.abs(packet_velocity_km_s - winner_velocity_km_s)
        / np.abs(winner_velocity_km_s) * 100
    ),
})
component_summary = pd.Series({
    'winner': winner['name'],
    'norad_id': winner_norad,
    'best_time_utc': winner_time.isoformat(),
    'position_error_km': np.linalg.norm(packet_position_km - winner_position_km),
    'velocity_error_m_s': np.linalg.norm(packet_velocity_km_s - winner_velocity_km_s) * 1000,
    'relative_position_error_percent': (
        np.linalg.norm(packet_position_km - winner_position_km)
        / np.linalg.norm(packet_position_km) * 100
    ),
    'relative_velocity_error_percent': (
        np.linalg.norm(packet_velocity_km_s - winner_velocity_km_s)
        / np.linalg.norm(packet_velocity_km_s) * 100
    ),
    'position_direction_angle_deg': vector_angle_deg(packet_position_km, winner_position_km),
    'velocity_direction_angle_deg': vector_angle_deg(packet_velocity_km_s, winner_velocity_km_s),
})
display(position_components)
display(velocity_components)
display(component_summary.to_frame('value'))


## 8. 速度权重敏感性

默认100 s只是把速度误差换算为等效位置误差的分析尺度，不是协议常数。对活动卫星目录中的全部唯一NORAD目标重复多组权重和精确时间优化，检查全目录第一名是否改变。


In [ ]:
SENSITIVITY_WEIGHTS_S = [0, 10, 30, 100, 300, 1000, 3000]
# 权重敏感性也不使用初筛候选池：每个权重都检查全部唯一NORAD目标。
sensitivity_pool = set(quick_rank['norad_id'].astype(int))

sensitivity_rows = []
for weight_s in SENSITIVITY_WEIGHTS_S:
    rows = []
    for number, norad in enumerate(sorted(sensitivity_pool), start=1):
        satellite = satellite_by_norad[int(norad)]
        result = precise_match(satellite, weight_s=weight_s)
        rows.append({
            'name': satellite.name or '',
            'norad_id': int(norad),
            **result,
        })
        if number % 1000 == 0:
            print(
                f'权重 {weight_s:g} s：已精确优化 '
                f'{number}/{len(sensitivity_pool)}'
            )
    ranking = pd.DataFrame(rows).sort_values('joint_score').reset_index(drop=True)
    assert len(ranking) == len(satellites), f'权重 {weight_s:g} s 未覆盖全部唯一NORAD目标'
    first = ranking.iloc[0]
    second = ranking.iloc[1]
    sensitivity_rows.append({
        'velocity_weight_s': weight_s,
        'catalogue_targets_tested': len(ranking),
        'winner': first['name'],
        'winner_norad': int(first['norad_id']),
        'best_offset_s': first['best_offset_s'],
        'position_error_km': first['best_position_error_km'],
        'velocity_error_m_s': first['best_velocity_error_km_s'] * 1000,
        'winner_joint_score_km2': first['joint_score'],
        'runner_up': second['name'],
        'runner_up_norad': int(second['norad_id']),
        'runner_up_best_offset_s': second['best_offset_s'],
        'runner_up_position_error_km': second['best_position_error_km'],
        'runner_up_velocity_error_m_s': second['best_velocity_error_km_s'] * 1000,
        'runner_up_joint_score_km2': second['joint_score'],
        'score_ratio_runner_to_winner': second['joint_score'] / first['joint_score'],
    })

sensitivity_table = pd.DataFrame(sensitivity_rows)
display(sensitivity_table)
print('所有权重冠军一致:', sensitivity_table['winner_norad'].nunique() == 1)


## 9. Event 2的差分多普勒一致性检验

Event 2没有可用状态向量，因此用Event 1得到的冠军和最佳TLE时间偏移预测两个事件的距离变化率。绝对CFO包含发射机、采样钟和中心频率误差；比较100 s内CFO变化可以抵消大部分公共偏置。


In [ ]:
def station_ecef_km(latitude_deg, longitude_deg, elevation_m):
    latitude = np.radians(latitude_deg)
    longitude = np.radians(longitude_deg)
    height_km = elevation_m / 1000.0
    a_km = 6378.137
    flattening = 1.0 / 298.257223563
    eccentricity_sq = flattening * (2.0 - flattening)
    n_km = a_km / np.sqrt(1.0 - eccentricity_sq * np.sin(latitude) ** 2)
    return np.asarray([
        (n_km + height_km) * np.cos(latitude) * np.cos(longitude),
        (n_km + height_km) * np.cos(latitude) * np.sin(longitude),
        (n_km * (1.0 - eccentricity_sq) + height_km) * np.sin(latitude),
    ])

def corrected_tle_geometry(satellite, reference_utc, tle_offset_s, site_ecef):
    model_utc = reference_utc + timedelta(seconds=float(tle_offset_s))
    position, velocity = satellite_state_ecef(
        satellite, ts.from_datetime(model_utc)
    )
    relative = position - site_ecef
    slant_range_km = float(np.linalg.norm(relative))
    range_rate_km_s = float(np.dot(relative, velocity) / slant_range_km)
    doppler_hz = -RF_FREQUENCY_HZ * range_rate_km_s / LIGHT_SPEED_KM_S
    return {
        'model_utc': model_utc.isoformat(),
        'slant_range_km': slant_range_km,
        'range_rate_km_s': range_rate_km_s,
        'predicted_doppler_hz': doppler_hz,
    }

site_ecef = station_ecef_km(
    SITE_LATITUDE_DEG, SITE_LONGITUDE_DEG, SITE_ELEVATION_M
)
event_reference_times = {
    'Event 1': gps_state_utc,
    'Event 2': gps_state_utc + timedelta(seconds=float(measured_spacing_s)),
}
geometry_rows = []
for label, reference_utc in event_reference_times.items():
    row = corrected_tle_geometry(
        winner_satellite, reference_utc, winner['best_offset_s'], site_ecef
    )
    observed_cfo = float(summaries[label]['packets'][0]['carrier_offset_hz'])
    row.update({
        'event': label,
        'reference_utc': reference_utc.isoformat(),
        'observed_cfo_hz': observed_cfo,
        'observed_minus_predicted_hz': observed_cfo - row['predicted_doppler_hz'],
    })
    geometry_rows.append(row)

event_geometry = pd.DataFrame(geometry_rows).set_index('event')
observed_change_hz = (
    event_geometry.loc['Event 2', 'observed_cfo_hz']
    - event_geometry.loc['Event 1', 'observed_cfo_hz']
)
predicted_change_hz = (
    event_geometry.loc['Event 2', 'predicted_doppler_hz']
    - event_geometry.loc['Event 1', 'predicted_doppler_hz']
)
doppler_summary = pd.Series({
    'observed_cfo_change_hz': observed_change_hz,
    'predicted_doppler_change_hz': predicted_change_hz,
    'change_residual_hz': observed_change_hz - predicted_change_hz,
})
display(event_geometry)
display(doppler_summary.to_frame('value'))


## 10. 研究结果可视化

本节生成五幅可独立用于论文排版的图，并同时保存PNG（300 dpi）和矢量PDF：

1. $W=100$ s时全部活动卫星的位置—速度误差分布；
2. 七组速度权重下第一名和第二名的位置—速度误差；所有标记均位于真实误差坐标，并用两级无刻度圆形放大框依次展示STARLINK-32838的五点密集组和其中的三点极密集组；
3. Event 1到Event 2的差分多普勒演化；


横纵轴及图例均明确标注单位。所有图片保存到`results/3_satellite_identification`；图中不写论文式长标题，详细说明留给图注。

图10.2的第二名随权重分为三组：$W=0,10$ s为STARLINK-11122 [DTC]，$W=30,100,300$ s为STARLINK-31335，$W=1000,3000$ s为STARLINK-5192。STARLINK-11122 [DTC]在$W=0$和$W=10$ s时完全重叠，是因为两次有界优化都到达搜索窗口的$+180$ s边界，从而取用了同一模型状态；这不是两个独立极小值恰好一致。第一级圆形放大框仅包含$W=0$至$300$ s的五个密集点，第二级继续放大其中$W=0,10,30$ s的三个极密集点；$W=1000,3000$ s在主图已经可分辨，不进入放大框。两级放大均保持真实线性坐标关系，精确数值以CSV表为准。


In [ ]:
from matplotlib.lines import Line2D

FIGURE_DIR = OUTPUT_DIR 
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DPI = 600
# 离散高对比配色：避免七组权重在印刷和缩放后看起来过于接近。
WEIGHT_COLORS = {
    0.0: '#D62728',     # red
    10.0: '#1F77B4',    # blue
    30.0: '#FF7F0E',    # orange
    100.0: '#2CA02C',   # green
    300.0: '#9467BD',   # purple
    1000.0: '#8C564B',  # brown
    3000.0: '#17BECF',  # cyan
}

def save_paper_figure(fig, stem):
    png_path = FIGURE_DIR / f'{stem}.png'
    pdf_path = FIGURE_DIR / f'{stem}.pdf'
    fig.savefig(png_path, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
    print('Saved:', png_path)
    print('Saved:', pdf_path)
    return png_path, pdf_path

# 新版第8节会直接保存第二名的完整指标；若当前内存仍是旧输出，
# 这里只重新优化每个权重对应的第二名，不重新运行全目录搜索。
sensitivity_plot_rows = []
has_runner_details = {
    'runner_up_norad', 'runner_up_best_offset_s',
    'runner_up_position_error_km', 'runner_up_velocity_error_m_s',
    'runner_up_joint_score_km2',
}.issubset(sensitivity_table.columns)

for row in sensitivity_table.itertuples(index=False):
    weight_s = float(row.velocity_weight_s)
    winner_score = (
        float(row.winner_joint_score_km2)
        if hasattr(row, 'winner_joint_score_km2')
        else score_from_errors(
            float(row.position_error_km),
            float(row.velocity_error_m_s) / 1000.0,
            weight_s,
        )
    )
    sensitivity_plot_rows.append({
        'velocity_weight_s': weight_s,
        'role': 'Winner',
        'name': row.winner,
        'norad_id': int(row.winner_norad),
        'best_offset_s': float(row.best_offset_s),
        'position_error_km': float(row.position_error_km),
        'velocity_error_m_s': float(row.velocity_error_m_s),
        'joint_score_km2': winner_score,
    })

    if has_runner_details:
        runner_values = {
            'name': row.runner_up,
            'norad_id': int(row.runner_up_norad),
            'best_offset_s': float(row.runner_up_best_offset_s),
            'position_error_km': float(row.runner_up_position_error_km),
            'velocity_error_m_s': float(row.runner_up_velocity_error_m_s),
            'joint_score_km2': float(row.runner_up_joint_score_km2),
        }
    else:
        matches = [sat for sat in satellites if (sat.name or '') == row.runner_up]
        assert len(matches) == 1, (row.runner_up, len(matches))
        runner_satellite = matches[0]
        runner_result = precise_match(runner_satellite, weight_s=weight_s)
        runner_values = {
            'name': row.runner_up,
            'norad_id': int(runner_satellite.model.satnum),
            'best_offset_s': float(runner_result['best_offset_s']),
            'position_error_km': float(runner_result['best_position_error_km']),
            'velocity_error_m_s': float(runner_result['best_velocity_error_km_s'] * 1000),
            'joint_score_km2': float(runner_result['joint_score']),
        }
    sensitivity_plot_rows.append({
        'velocity_weight_s': weight_s,
        'role': 'Runner-up',
        **runner_values,
    })

sensitivity_top2 = pd.DataFrame(sensitivity_plot_rows)
sensitivity_top2.to_csv(FIGURE_DIR / 'sensitivity_top2_details.csv', index=False)
display(sensitivity_top2)


In [ ]:
# Figure 10.1: W=100 s，全活动卫星目录的位置—速度误差分布。
plot_rank = final_rank.copy()
plot_rank['velocity_error_m_s'] = plot_rank['best_velocity_error_km_s'] * 1000.0
plot_names = plot_rank['name'].fillna('').astype(str)
is_starlink = plot_names.str.upper().str.startswith('STARLINK')
is_target = plot_rank['norad_id'].astype(int) == winner_norad
runner_row = plot_rank.iloc[1]
is_runner = plot_rank['norad_id'].astype(int) == int(runner_row['norad_id'])
is_other_starlink = is_starlink & ~is_target & ~is_runner
is_non_starlink = ~is_starlink & ~is_target & ~is_runner
assert (
    int(is_non_starlink.sum()) + int(is_other_starlink.sum())
    + int(is_target.sum()) + int(is_runner.sum())
    == len(plot_rank)
), '卫星分类计数与全目录数量不一致'

fig, ax = plt.subplots(figsize=(8.2, 5.6), constrained_layout=True)
ax.scatter(
    plot_rank.loc[is_non_starlink, 'best_position_error_km'],
    plot_rank.loc[is_non_starlink, 'velocity_error_m_s'],
    s=10, c='#9b7ede', alpha=0.18, linewidths=0, rasterized=True,
)
ax.scatter(
    plot_rank.loc[is_other_starlink, 'best_position_error_km'],
    plot_rank.loc[is_other_starlink, 'velocity_error_m_s'],
    s=10, c='#2878b5', alpha=0.58, linewidths=0, rasterized=True,
)
ax.scatter(
    winner['best_position_error_km'], winner['best_velocity_error_km_s'] * 1000,
    marker='*', s=280, c='#d62728', edgecolors='black', linewidths=0.8,
    zorder=6,
)
ax.scatter(
    runner_row['best_position_error_km'], runner_row['velocity_error_m_s'],
    marker='^', s=95, c='#ff9f1c', edgecolors='black', linewidths=0.7,
    zorder=5,
)
ax.annotate(
    'STARLINK-32838',
    (winner['best_position_error_km'], winner['best_velocity_error_km_s'] * 1000),
    xytext=(9, 8), textcoords='offset points', fontsize=9,
)
ax.annotate(
    str(runner_row['name']),
    (runner_row['best_position_error_km'], runner_row['velocity_error_m_s']),
    xytext=(9, 8), textcoords='offset points', fontsize=9,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Position error (km)')
ax.set_ylabel('Velocity error (m/s)')
ax.grid(True, which='major', color='#aeb6bf', alpha=0.42, linewidth=0.70)
ax.grid(False, which='minor')
ax.tick_params(which='major', length=6, width=0.8)
ax.tick_params(which='minor', length=3, width=0.5)
legend_circle_size = 6
legend_star_size = 9
legend_triangle_size = 7
legend_handles = [
    Line2D([], [], marker='*', linestyle='None', markersize=legend_star_size,
           markerfacecolor='#d62728', markeredgecolor='black', markeredgewidth=0.7,
           label='STARLINK-32838'),
    Line2D([], [], marker='^', linestyle='None', markersize=legend_triangle_size,
           markerfacecolor='#ff9f1c', markeredgecolor='black', markeredgewidth=0.7,
           label=f"Runner-up: {runner_row['name']}"),
    Line2D([], [], marker='o', linestyle='None', markersize=legend_circle_size,
           markerfacecolor='#9b7ede', markeredgewidth=0, alpha=0.75,
           label=f'Other active satellites ({is_non_starlink.sum():,})'),
    Line2D([], [], marker='o', linestyle='None', markersize=legend_circle_size,
           markerfacecolor='#2878b5', markeredgewidth=0, alpha=0.80,
           label=f'Other Starlink satellites ({is_other_starlink.sum():,})'),
]
ax.legend(handles=legend_handles, loc='best', fontsize=8, frameon=True)
save_paper_figure(fig, 'figure10_1_all_satellites_w100')
plt.show()


In [ ]:
# Figure 10.2: 七组权重下，最佳与次佳候选的位置—速度误差。
# 所有主图标记均使用真实误差坐标；圆形放大框只展示第一名聚类。
from matplotlib.patches import Circle, ConnectionPatch

fig, ax = plt.subplots(figsize=(8.4, 4.5), constrained_layout=True)
for weight_s in SENSITIVITY_WEIGHTS_S:
    subset = sensitivity_top2[sensitivity_top2['velocity_weight_s'] == weight_s]
    first = subset[subset['role'] == 'Winner'].iloc[0]
    second = subset[subset['role'] == 'Runner-up'].iloc[0]
    color = WEIGHT_COLORS[weight_s]
    ax.scatter(
        first['position_error_km'], first['velocity_error_m_s'],
        marker='*', s=110, color=color, edgecolor='black', linewidth=0.65,
        zorder=4,
    )
    ax.scatter(
        second['position_error_km'], second['velocity_error_m_s'],
        marker='^', s=55, color=color, edgecolor='black', linewidth=0.6,
        zorder=3,
    )

runner_points = sensitivity_top2[sensitivity_top2['role'] == 'Runner-up']
runner_label_style = {
    'STARLINK-11122 [DTC]': {'xytext': (8, -14), 'ha': 'left'},
    'STARLINK-31335': {'xytext': (0, 14), 'ha': 'center'},
    'STARLINK-5192': {'xytext': (-8, 10), 'ha': 'right'},
}
for runner_name, group in runner_points.groupby('name', sort=False):
    label_row = group.iloc[len(group) // 2]
    label_style = runner_label_style.get(
        runner_name, {'xytext': (8, 8), 'ha': 'left'}
    )
    ax.annotate(
        runner_name,
        (label_row['position_error_km'], label_row['velocity_error_m_s']),
        xytext=label_style['xytext'], textcoords='offset points',
        ha=label_style['ha'], fontsize=8,
    )
winner_points = sensitivity_top2[sensitivity_top2['role'] == 'Winner']
winner_label_row = winner_points.loc[winner_points['position_error_km'].idxmax()]
ax.annotate(
    'STARLINK-32838',
    (winner_label_row['position_error_km'], winner_label_row['velocity_error_m_s']),
    xytext=(12, 4), textcoords='offset points', fontsize=8,
)

role_legend = ax.legend(
    handles=[
        Line2D([], [], marker='*', linestyle='None', markersize=11,
               markerfacecolor='white', markeredgecolor='black',
               label='Best candidate'),
        Line2D([], [], marker='^', linestyle='None', markersize=8,
               markerfacecolor='white', markeredgecolor='black',
               label='Second-best candidate'),
    ],
    loc='upper left', fontsize=8,
)
ax.add_artist(role_legend)
weight_handles = [
    Line2D([], [], marker='o', linestyle='None', color=WEIGHT_COLORS[w],
           markersize=5, label=f'{w:g} s')
    for w in SENSITIVITY_WEIGHTS_S
]
ax.legend(
    handles=weight_handles, loc='lower right', ncol=2, fontsize=8,
    title='Velocity weight W', title_fontsize=8,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(4.8, 260.0)
ax.set_ylim(9.5, 9000.0)
ax.set_xlabel('Position error (km)', fontsize=10)
ax.set_ylabel('Velocity error (m/s)', fontsize=10)
ax.tick_params(axis='both', which='major', labelsize=10)
ax.grid(True, which='major', color='#aeb6bf', alpha=0.35, linewidth=0.7)
ax.grid(False, which='minor')

# 两级无刻度圆形放大框；只显示主图中无法分辨的真实坐标点。
dense_five = winner_points[winner_points['velocity_weight_s'] <= 300.0]
dense_three = winner_points[winner_points['velocity_weight_s'] <= 30.0]

# 第一级：W=0--300 s五点。W=1000和3000 s在主图中已经可分辨。
zoom1_ax = ax.inset_axes([0.11, 0.18, 0.29, 0.29], zorder=6)
zoom1_ax.set_box_aspect(1)
zoom1_ax.set_xlim(5.466, 5.494)
zoom1_ax.set_ylim(13.24, 13.49)
zoom1_ax.set_axis_off()
zoom1_ax.patch.set_alpha(0.0)
lens1_circle = Circle(
    (0.5, 0.5), 0.48, transform=zoom1_ax.transAxes,
    facecolor='white', edgecolor='#4d4d4d', linewidth=1.0, zorder=0,
)
zoom1_ax.add_patch(lens1_circle)
for _, row in dense_five.iterrows():
    point = zoom1_ax.scatter(
        row['position_error_km'], row['velocity_error_m_s'],
        marker='*', s=105,
        color=WEIGHT_COLORS[row['velocity_weight_s']],
        edgecolor='black', linewidth=0.6, zorder=3,
    )
    point.set_clip_path(lens1_circle)
zoom1_ax.text(
    0.48, 0.48, r'$\rm W \leq 300~\mathrm{s}$', transform=zoom1_ax.transAxes,
    ha='center', va='center', fontsize=8, zorder=4,
)

# 第二级：继续放大第一级左上方的W=0,10,30 s三点。
zoom2_ax = ax.inset_axes([0.42, 0.51, 0.22, 0.22], zorder=7)
zoom2_ax.set_box_aspect(1)
zoom2_ax.set_xlim(5.4724628, 5.4724662)
zoom2_ax.set_ylim(13.4330, 13.4361)
zoom2_ax.set_axis_off()
zoom2_ax.patch.set_alpha(0.0)
lens2_circle = Circle(
    (0.5, 0.5), 0.48, transform=zoom2_ax.transAxes,
    facecolor='white', edgecolor='#4d4d4d', linewidth=1.0, zorder=0,
)
zoom2_ax.add_patch(lens2_circle)
for _, row in dense_three.iterrows():
    point = zoom2_ax.scatter(
        row['position_error_km'], row['velocity_error_m_s'],
        marker='*', s=100,
        color=WEIGHT_COLORS[row['velocity_weight_s']],
        edgecolor='black', linewidth=0.6, zorder=3,
    )
    point.set_clip_path(lens2_circle)
zoom2_ax.text(
    0.50, 0.55, r'$\rm W \leq 30~\mathrm{s}$', transform=zoom2_ax.transAxes,
    ha='center', va='center', fontsize=7, zorder=4,
)

# 主图五点密集组 -> 第一级放大。
fig.canvas.draw()
dense_five_center = (
    float(dense_five['position_error_km'].median()),
    float(dense_five['velocity_error_m_s'].median()),
)
dense_five_center_axes = ax.transAxes.inverted().transform(
    ax.transData.transform(dense_five_center)
)
ax.scatter(
    [dense_five_center[0]], [dense_five_center[1]],
    marker='o', s=420, facecolors='none', edgecolors='#4d4d4d',
    linewidths=0.9, zorder=5,
)
main_to_zoom1 = [
    ((-0.008, 0.018), (0.12, 0.79)),  # 小圆上缘 -> 第一级圆左上缘
    ((0.010, -0.018), (0.79, 0.12)),  # 小圆下缘 -> 第一级圆右下缘
]
for (source_dx, source_dy), lens_point in main_to_zoom1:
    fig.add_artist(ConnectionPatch(
        xyA=(dense_five_center_axes[0] + source_dx,
             dense_five_center_axes[1] + source_dy),
        coordsA='axes fraction', axesA=ax,
        xyB=lens_point, coordsB='axes fraction', axesB=zoom1_ax,
        color='#666666', linewidth=0.8, alpha=0.82, zorder=2,
    ))

# 第一级中的三点极密集组 -> 第二级放大。
dense_three_center = (
    float(dense_three['position_error_km'].median()),
    float(dense_three['velocity_error_m_s'].median()),
)
dense_three_center_axes = zoom1_ax.transAxes.inverted().transform(
    zoom1_ax.transData.transform(dense_three_center)
)
zoom1_ax.scatter(
    [dense_three_center[0]], [dense_three_center[1]],
    marker='o', s=120, facecolors='none', edgecolors='#4d4d4d',
    linewidths=0.8, zorder=5,
)
zoom1_to_zoom2 = [
    ((-0.035, 0.035), (0.14, 0.82)),  # 三级密集组上缘 -> 第二级圆左上缘
    ((0.035, -0.035), (0.50, 0.02)),  # 三级密集组下缘 -> 第二级圆底缘
]
for (source_dx, source_dy), lens_point in zoom1_to_zoom2:
    fig.add_artist(ConnectionPatch(
        xyA=(dense_three_center_axes[0] + source_dx,
             dense_three_center_axes[1] + source_dy),
        coordsA='axes fraction', axesA=zoom1_ax,
        xyB=lens_point, coordsB='axes fraction', axesB=zoom2_ax,
        color='#666666', linewidth=0.8, alpha=0.82, zorder=3,
    ))

save_paper_figure(fig, 'figure10_2_weight_top2_error_plane')
plt.show()


In [ ]:
# Figure 10.4: 以Event 1为零点的预测与观测差分多普勒。
doppler_times_s = np.linspace(0.0, float(measured_spacing_s), 320)
predicted_doppler_curve_hz = np.asarray([
    corrected_tle_geometry(
        winner_satellite,
        gps_state_utc + timedelta(seconds=float(delta_s)),
        winner['best_offset_s'],
        site_ecef,
    )['predicted_doppler_hz']
    for delta_s in doppler_times_s
])
predicted_relative_hz = predicted_doppler_curve_hz - predicted_doppler_curve_hz[0]
event_times_s = np.asarray([0.0, float(measured_spacing_s)])
observed_relative_hz = np.asarray([0.0, float(observed_change_hz)])
predicted_event2_hz = float(predicted_relative_hz[-1])

# --- 从原始电压重新估计两个事件的实测CFO及其统计不确定度 ---
# 这里复用LFdemod已有DDC与LoRa重采样，不修改LFdemod。为了与原始解调完全一致，
# 每个事件仍读取summary记录的完整padding窗口；这一步会重新处理约1.6--1.8 s原始电压。
from lora_phy import LoRaReceiver
from lfdemod.starlink_vhf import (
    StarlinkVHFDDCConfig, StarlinkVHFLoRaConfig,
    downconvert_starlink_vhf_voltage, prepare_lora_iq,
)

CFO_BOOTSTRAP_DRAWS = 10_000
CFO_TIMING_SEARCH_SAMPLES = 0
CFO_RANDOM_SEED = 32_838

def _one_dechirped_cfo(receiver, filtered_iq, left):
    """Return one preamble chirp's quantized CFO and peak concentration."""
    samples = filtered_iq[left:left + receiver._sample_num]
    if samples.size != receiver._sample_num:
        raise ValueError('Preamble chirp extends beyond the available IQ window')
    spectrum = np.fft.fft(
        samples * receiver._down_chirp[:samples.size],
        receiver._fft_len,
    )
    combined = (
        np.abs(spectrum[:receiver._bin_num])
        + np.abs(spectrum[-receiver._bin_num:])
    )
    peak_bin = int(np.argmax(combined))
    signed_bin = (
        peak_bin - receiver._bin_num
        if peak_bin + 1 > receiver._bin_num / 2
        else peak_bin
    )
    cfo_hz = signed_bin * receiver._bandwidth / receiver._bin_num
    tiny = np.finfo(float).tiny
    concentration_db = 20.0 * np.log10(
        max(float(combined[peak_bin]), tiny)
        / max(float(np.median(combined)), tiny)
    )
    return float(cfo_hz), float(concentration_db)

def estimate_event_preamble_cfo(label, random_seed):
    """Estimate CFO from all 15 ordinary preamble chirps of one event."""
    summary = summaries[label]
    packet = summary['packets'][0]
    raw_path = Path(summary['input_path'])
    dtype = np.dtype(summary['config']['dtype'])
    ddc_config = StarlinkVHFDDCConfig(**summary['config']['ddc'])
    lora_values = dict(summary['config']['lora'])
    lora_values['accepted_payload_bytes'] = tuple(
        lora_values['accepted_payload_bytes']
    )
    lora_config = StarlinkVHFLoRaConfig(**lora_values)

    file_samples = raw_path.stat().st_size // dtype.itemsize
    file_duration_s = file_samples / ddc_config.fs_in
    padded_start_s = max(
        0.0, summary['requested_start_seconds'] - summary['padding_seconds']
    )
    padded_stop_s = min(
        file_duration_s,
        summary['requested_start_seconds']
        + summary['requested_duration_seconds']
        + summary['padding_seconds'],
    )
    start_sample = int(round(padded_start_s * ddc_config.fs_in))
    stop_sample = int(round(padded_stop_s * ddc_config.fs_in))
    initial_phase = float(np.remainder(
        2.0 * np.pi * summary['config']['rf_frequency_hz']
        * start_sample / ddc_config.fs_in,
        2.0 * np.pi,
    ))
    raw = np.memmap(raw_path, dtype=dtype, mode='r')
    channel_iq, _ = downconvert_starlink_vhf_voltage(
        raw[start_sample:stop_sample],
        summary['config']['rf_frequency_hz'],
        ddc_config,
        initial_phase,
    )
    del raw
    lora_iq, lora_rate_hz = prepare_lora_iq(
        channel_iq, ddc_config.fs_out, lora_config
    )
    receiver = LoRaReceiver(
        summary['config']['rf_frequency_hz'],
        lora_config.spreading_factor,
        lora_config.bandwidth_hz,
        lora_rate_hz,
        has_header=lora_config.has_explicit_header,
        implicit_header_payload_len=lora_config.expected_payload_bytes,
        implicit_header_coding_rate=lora_config.expected_coding_rate,
        implicit_header_enable_crc=lora_config.expected_crc_enabled,
        preamble_len=lora_config.preamble_symbols,
        resample_to_2x=False,
    )
    filtered_iq = np.asarray(receiver.lowpass(lora_iq))
    nominal_start = int(round(
        (packet['preamble_start_in_file_seconds'] - padded_start_s)
        * lora_rate_hz
    ))

    # summary中的preamble时刻已经由完整LoRa同步得到，因此固定使用该起点。
    # 不再移动LoRa采样点，避免chirp的时间—频率耦合改变CFO估计。
    timing_candidates = []
    for shift in range(-CFO_TIMING_SEARCH_SAMPLES, CFO_TIMING_SEARCH_SAMPLES + 1):
        cfo_values = []
        concentration_values = []
        for chirp_index in range(lora_config.preamble_symbols):
            left = (
                nominal_start + shift
                + chirp_index * receiver._sample_num
            )
            cfo_hz, concentration_db = _one_dechirped_cfo(
                receiver, filtered_iq, left
            )
            cfo_values.append(cfo_hz)
            concentration_values.append(concentration_db)
        timing_candidates.append({
            'shift_samples': shift,
            'cfo_hz': np.asarray(cfo_values, dtype=float),
            'concentration_db': np.asarray(concentration_values, dtype=float),
            'median_concentration_db': float(np.median(concentration_values)),
        })
    best_timing = max(
        timing_candidates,
        key=lambda row: (row['median_concentration_db'], -abs(row['shift_samples'])),
    )
    cfo_values = best_timing['cfo_hz']
    cfo_median_hz = float(np.median(cfo_values))

    # Bootstrap对15个chirp有放回重采样，得到中位数估计器的经验散布。
    rng = np.random.default_rng(random_seed)
    draw_indices = rng.integers(
        0, cfo_values.size, size=(CFO_BOOTSTRAP_DRAWS, cfo_values.size)
    )
    bootstrap_medians_hz = np.median(cfo_values[draw_indices], axis=1)
    bootstrap_q16_hz, bootstrap_q84_hz = np.percentile(
        bootstrap_medians_hz, [16.0, 84.0]
    )
    bootstrap_sigma_hz = float(
        0.5 * (bootstrap_q84_hz - bootstrap_q16_hz)
    )
    # lora-phy使用10倍零填充；离散FFT网格间隔为BW/(2^SF*10)。
    # 把一个网格内均匀未知的位置换算为标准差delta_f/sqrt(12)。
    fft_grid_hz = float(receiver._bandwidth / receiver._bin_num)
    grid_sigma_hz = fft_grid_hz / np.sqrt(12.0)
    # 保守地把15个chirp之间的全部样本标准差作为单事件重复性误差；
    # 不除以sqrt(N)，因此它不是中位数标准误，而是经验重复性上限。
    chirp_repeatability_sigma_hz = float(np.std(cfo_values, ddof=1))
    total_sigma_hz = float(np.hypot(
        chirp_repeatability_sigma_hz, grid_sigma_hz
    ))

    chirp_rows = pd.DataFrame({
        'event': label,
        'chirp_number': np.arange(1, lora_config.preamble_symbols + 1),
        'cfo_hz': cfo_values,
        'concentration_db': best_timing['concentration_db'],
    })
    result = {
        'event': label,
        'preamble_chirps': lora_config.preamble_symbols,
        'timing_shift_samples': best_timing['shift_samples'],
        'timing_shift_us': best_timing['shift_samples'] / lora_rate_hz * 1e6,
        'median_concentration_db': best_timing['median_concentration_db'],
        'backend_single_chirp_cfo_hz': packet['carrier_offset_hz'],
        'preamble_median_cfo_hz': cfo_median_hz,
        'bootstrap_q16_hz': bootstrap_q16_hz,
        'bootstrap_q84_hz': bootstrap_q84_hz,
        'bootstrap_sigma_hz': bootstrap_sigma_hz,
        'chirp_repeatability_sigma_hz': chirp_repeatability_sigma_hz,
        'fft_grid_hz': fft_grid_hz,
        'grid_sigma_hz': grid_sigma_hz,
        'total_sigma_hz': total_sigma_hz,
    }
    return result, chirp_rows

cfo_results = []
cfo_chirp_tables = []
for event_number, label in enumerate(('Event 1', 'Event 2')):
    result, chirp_table = estimate_event_preamble_cfo(
        label, CFO_RANDOM_SEED + event_number
    )
    cfo_results.append(result)
    cfo_chirp_tables.append(chirp_table)

cfo_uncertainty = pd.DataFrame(cfo_results).set_index('event')
cfo_per_chirp = pd.concat(cfo_chirp_tables, ignore_index=True)
observed_delta_cfo_hz = (
    cfo_uncertainty.loc['Event 2', 'preamble_median_cfo_hz']
    - cfo_uncertainty.loc['Event 1', 'preamble_median_cfo_hz']
)
observed_delta_sigma_hz = float(np.hypot(
    cfo_uncertainty.loc['Event 1', 'total_sigma_hz'],
    cfo_uncertainty.loc['Event 2', 'total_sigma_hz'],
))
display(cfo_uncertainty)
display(cfo_per_chirp)
print(
    f'Observed Event 2 - Event 1 CFO: {observed_delta_cfo_hz:.3f} '
    f'+/- {observed_delta_sigma_hz:.3f} Hz (conservative empirical uncertainty)'
)
print(f'Paper value: Delta f_obs = {observed_delta_cfo_hz:.2f} +/- {observed_delta_sigma_hz:.2f} Hz')
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cfo_uncertainty.to_csv(OUTPUT_DIR / 'observed_cfo_uncertainty.csv')
    cfo_per_chirp.to_csv(
        OUTPUT_DIR / 'preamble_cfo_per_chirp.csv', index=False
    )

fig, ax = plt.subplots(figsize=(8.4, 5.5), constrained_layout=True)
ax.plot(
    doppler_times_s, predicted_relative_hz, color='#2878b5', linewidth=2.0,
    label='STARLINK-32838 TLE prediction',
)
ax.scatter(
    event_times_s, observed_relative_hz, marker='*', s=170,
    color='#d62728', edgecolor='black', linewidth=0.7, zorder=4,
    label='Observed LoRa CFO change',
)
ax.plot(
    [measured_spacing_s, measured_spacing_s],
    [predicted_event2_hz, observed_change_hz],
    linestyle='--', color='0.25', linewidth=1.1,
)
ax.annotate(
    f'Residual = {observed_change_hz - predicted_event2_hz:+.1f} Hz',
    (measured_spacing_s, (predicted_event2_hz + observed_change_hz) / 2),
    xytext=(-8, 10), textcoords='offset points', ha='right', fontsize=9,
)
ax.axhline(0, color='0.55', linewidth=0.7)
ax.set_xlabel('Time relative to Event 1 preamble (s)')
ax.set_ylabel(r'Frequency change relative to Event 1 (Hz)')
ax.grid(True, alpha=0.22, linewidth=0.6)
ax.legend(loc='best', fontsize=8)
save_paper_figure(fig, 'figure10_4_differential_doppler')
plt.show()


In [ ]:
# 图3补充：事件前历史TLE在各自最佳时间偏移下的差分多普勒敏感性包络。
# 这不是统计置信区间：每条曲线先用Event 1的payload六维状态和W=100 s独立对齐，
# 再计算从Event 1到Event 2约100 s内的相对多普勒变化。
# 实测点由每个事件15个普通preamble upchirp独立估计CFO；Event 2误差棒结合
# chirp间经验重复性（样本标准差）与LoRa解调FFT频率网格的量化下限。
HISTORICAL_TLE_PATH = (
    DATA_DIR / 'orbital' / 'STARLINK-32838_2025-04-10_2025-04-20.tle'
)
assert HISTORICAL_TLE_PATH.is_file(), HISTORICAL_TLE_PATH

with load.open(str(HISTORICAL_TLE_PATH)) as tle_file:
    historical_target_tles = [
        sat for sat in parse_tle_file(tle_file, ts)
        if int(sat.model.satnum) == winner_norad
    ]

# 只按真正参与SGP4传播的动力学参数去重；同一epoch但动力学参数不同的修订解仍保留。
def tle_dynamical_key(sat):
    model = sat.model
    return (
        float(sat.epoch.tt),
        float(model.inclo), float(model.nodeo), float(model.ecco),
        float(model.argpo), float(model.mo), float(model.no_kozai),
        float(model.bstar), float(model.ndot), float(model.nddot),
    )

unique_historical_target_tles = []
seen_dynamical_keys = set()
for sat in historical_target_tles:
    key = tle_dynamical_key(sat)
    if key not in seen_dynamical_keys:
        seen_dynamical_keys.add(key)
        unique_historical_target_tles.append(sat)

# 只用事件前已经发布的TLE建立主包络。事件后的TLE可能包含未知轨道机动，
# 向事件时刻反推时具有不同物理含义，因此不混入这条包络。
pre_event_tles = [
    sat for sat in unique_historical_target_tles
    if sat.epoch.utc_datetime() < gps_state_utc
]
assert pre_event_tles, '历史文件中没有找到事件前的目标TLE'

OFFSET_BOUND_MARGIN_S = 1.0
MAX_POSITION_ERROR_KM = 50.0
MAX_VELOCITY_ERROR_KM_S = 0.1

envelope_times_s = np.linspace(0.0, float(measured_spacing_s), 320)
pre_event_curves_hz = []
valid_pre_event_tles = []
pre_event_alignment_rows = []

for sat in pre_event_tles:
    # 每条历史TLE均独立搜索±TIME_BOUND_S内的最佳时刻；W固定为论文默认100 s。
    match = precise_match(sat, weight_s=VELOCITY_WEIGHT_S)
    absolute_doppler_hz = np.asarray([
        corrected_tle_geometry(
            sat,
            gps_state_utc + timedelta(seconds=float(relative_s)),
            match['best_offset_s'],
            site_ecef,
        )['predicted_doppler_hz']
        for relative_s in envelope_times_s
    ])
    relative_doppler_hz = absolute_doppler_hz - absolute_doppler_hz[0]

    offset_is_interior = (
        abs(match['best_offset_s']) < TIME_BOUND_S - OFFSET_BOUND_MARGIN_S
    )
    position_match_is_valid = (
        match['best_position_error_km'] < MAX_POSITION_ERROR_KM
    )
    velocity_match_is_valid = (
        match['best_velocity_error_km_s'] < MAX_VELOCITY_ERROR_KM_S
    )
    values_are_finite = bool(
        np.isfinite(match['best_offset_s'])
        and np.isfinite(match['best_position_error_km'])
        and np.isfinite(match['best_velocity_error_km_s'])
        and np.all(np.isfinite(relative_doppler_hz))
    )
    exclusion_reasons = []
    if not offset_is_interior:
        exclusion_reasons.append('offset_at_search_boundary')
    if not position_match_is_valid:
        exclusion_reasons.append('position_mismatch')
    if not velocity_match_is_valid:
        exclusion_reasons.append('velocity_mismatch')
    if not values_are_finite:
        exclusion_reasons.append('non_finite_propagation')
    valid_for_envelope = not exclusion_reasons

    if valid_for_envelope:
        valid_pre_event_tles.append(sat)
        pre_event_curves_hz.append(relative_doppler_hz)
    pre_event_alignment_rows.append({
        'tle_epoch_utc': sat.epoch.utc_datetime().isoformat(),
        'tle_age_at_event_days': (
            gps_state_utc - sat.epoch.utc_datetime()
        ).total_seconds() / 86400.0,
        'best_offset_s': match['best_offset_s'],
        'position_error_km': match['best_position_error_km'],
        'velocity_error_m_s': match['best_velocity_error_km_s'] * 1000.0,
        'event2_relative_doppler_hz': relative_doppler_hz[-1],
        'offset_is_interior': offset_is_interior,
        'position_match_is_valid': position_match_is_valid,
        'velocity_match_is_valid': velocity_match_is_valid,
        'values_are_finite': values_are_finite,
        'valid_for_envelope': valid_for_envelope,
        'exclusion_reasons': '; '.join(exclusion_reasons),
    })

assert pre_event_curves_hz, '没有历史TLE通过包络质量过滤'
pre_event_curves_hz = np.vstack(pre_event_curves_hz)
envelope_lower_hz = np.min(pre_event_curves_hz, axis=0)
envelope_upper_hz = np.max(pre_event_curves_hz, axis=0)

# 最新的事件前TLE作为名义中心曲线；包络仍由全部事件前TLE的逐时刻极值定义。
nominal_index = max(
    range(len(valid_pre_event_tles)),
    key=lambda index: valid_pre_event_tles[index].epoch.utc_datetime(),
)
nominal_curve_hz = pre_event_curves_hz[nominal_index]
pre_event_alignment = pd.DataFrame(pre_event_alignment_rows).sort_values(
    'tle_epoch_utc'
).reset_index(drop=True)
display(pre_event_alignment)
print('Historical target TLE records:', len(historical_target_tles))
print('Unique historical dynamical solutions:', len(unique_historical_target_tles))
print('Unique pre-event TLE solutions checked:', len(pre_event_tles))
print('Valid pre-event TLE curves in envelope:', len(valid_pre_event_tles))
print('Excluded pre-event solutions:', len(pre_event_tles) - len(valid_pre_event_tles))
print('Nominal latest pre-event epoch:', valid_pre_event_tles[nominal_index].epoch.utc_datetime().isoformat())
print(
    'Event 2 envelope range (Hz):',
    f'{envelope_lower_hz[-1]:.3f} to {envelope_upper_hz[-1]:.3f}',
)

fig, ax = plt.subplots(figsize=(8.4, 4.5), constrained_layout=True)
for curve_hz in pre_event_curves_hz:
    ax.plot(
        envelope_times_s, curve_hz,
        color='#74a9cf', alpha=0.20, linewidth=0.8, zorder=1,
        solid_capstyle='butt',
    )
ax.fill_between(
    envelope_times_s, envelope_lower_hz, envelope_upper_hz,
    color='#9ecae1', alpha=0.55, linewidth=0, edgecolor='none',
    antialiased=False, zorder=2,
    label=f'Valid pre-event TLE sensitivity envelope (N={len(valid_pre_event_tles)})',
)
ax.plot(
    envelope_times_s, nominal_curve_hz,
    color='#08519c', linewidth=2.0, zorder=3,
    solid_capstyle='butt',
    label='Latest pre-event TLE (nominal)',
)
# Event 1是人为定义的差分零点，用五角星强调reference；Event 2保留圆点。
ax.scatter(
    [0.0], [0.0], marker='*', s=115,
    color='#cb181d', edgecolor='black', linewidth=0.6,
    zorder=6, label='Event 1 reference',
)
ax.scatter(
    [float(measured_spacing_s)], [observed_delta_cfo_hz],
    marker='o', s=32, color='#cb181d',
    edgecolor='white', linewidth=0.7, zorder=6,
    label='Event 2 observed CFO difference',
)
# Event 2误差棒已经联合Event 1和Event 2的经验CFO不确定度。
ax.errorbar(
    [float(measured_spacing_s)], [observed_delta_cfo_hz],
    yerr=[observed_delta_sigma_hz], fmt='none',
    ecolor='#cb181d', elinewidth=1.2, capsize=4.0, capthick=1.2,
    zorder=4,
)
ax.axhline(0.0, color='0.55', linewidth=0.7)
# 横轴两端保留少量空白；曲线数据本身仍严格从tau=0开始。
x_padding_s = max(4.0, 0.04 * float(measured_spacing_s))
ax.set_xlim(-x_padding_s, float(measured_spacing_s) + x_padding_s)
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'Frequency offset (Hz)')
ax.grid(True, alpha=0.22, linewidth=0.6)
ax.legend(loc='best', fontsize=8)

# Event 2局部放大框：只改变坐标显示尺度，误差棒数值本身不放大。
zoom_window_s = 1.2
zoom_left_s = float(measured_spacing_s) - zoom_window_s
zoom_right_s = float(measured_spacing_s) + 0.12
zoom_mask = envelope_times_s >= zoom_left_s
zoom_y_values = np.concatenate([
    envelope_lower_hz[zoom_mask],
    envelope_upper_hz[zoom_mask],
    np.asarray([
        observed_delta_cfo_hz - observed_delta_sigma_hz,
        observed_delta_cfo_hz + observed_delta_sigma_hz,
    ]),
])
zoom_y_padding_hz = max(8.0, 0.10 * float(np.ptp(zoom_y_values)))

inset_ax = ax.inset_axes([0.54, 0.52, 0.41, 0.38])
inset_ax.fill_between(
    envelope_times_s, envelope_lower_hz, envelope_upper_hz,
    color='#9ecae1', alpha=0.55, linewidth=0, edgecolor='none',
    antialiased=False, zorder=1,
)
inset_ax.plot(
    envelope_times_s, nominal_curve_hz,
    color='#08519c', linewidth=1.5, solid_capstyle='butt', zorder=2,
)
inset_ax.errorbar(
    [float(measured_spacing_s)], [observed_delta_cfo_hz],
    yerr=[observed_delta_sigma_hz], fmt='o', markersize=4.6,
    color='#cb181d', markeredgecolor='white', markeredgewidth=0.6,
    ecolor='#cb181d', elinewidth=1.25, capsize=4.0, capthick=1.25,
    zorder=4,
)
inset_ax.set_xlim(zoom_left_s, zoom_right_s)
inset_ax.set_ylim(
    float(np.min(zoom_y_values) - zoom_y_padding_hz),
    float(np.max(zoom_y_values) + zoom_y_padding_hz),
)
inset_ax.set_title('Event 2 zoom', fontsize=8, pad=3)
inset_ax.tick_params(axis='both', labelsize=7)
inset_ax.grid(True, alpha=0.18, linewidth=0.5)
ax.indicate_inset_zoom(
    inset_ax, edgecolor='0.35', alpha=0.75, linewidth=0.8
)
save_paper_figure(fig, 'figure10_4b_pre_event_tle_sensitivity_envelope')
plt.show()


## 11. 保存完整的结构化验证结果

下一格把支撑目标识别的中间量、最终量、输入清单和方法说明统一保存到`results/3_satellite_identification`。

主要输出分为六组：

1. **帧与profile证据**：独立CRC、profile字段、两个包的关联字段；
2. **payload状态证据**：原始24-byte状态块逐分量解释、Event 1状态汇总；
3. **目录竞争证据**：全部卫星排名、冠军/第二名、六分量残差和权重敏感性；
4. **CFO与几何证据**：逐chirp CFO、两个事件的经验不确定度、站星几何和差分多普勒；
5. **历史TLE敏感性**：每条事件前TLE的对齐质量、全部有效曲线及上下包络；
6. **复现清单**：小型输入文件SHA-256、原始文件路径/大小/acquisition metadata、入选正文图和总JSON摘要。

六维分量结果继续保存，但仅作为诊断/补充材料。


In [ ]:
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # 1. 帧、profile和两个包之间的关系。
    frame_table.reset_index().to_csv(
        OUTPUT_DIR / 'frame_crc_validation.csv', index=False
    )
    profile_table.reset_index().to_csv(
        OUTPUT_DIR / 'starlink_profile_checks.csv', index=False
    )
    relationship_table.reset_index().to_csv(
        OUTPUT_DIR / 'packet_relationship.csv', index=False
    )
    packet_pair_summary.rename('value').to_csv(
        OUTPUT_DIR / 'packet_pair_summary.csv', header=True
    )

    # 2. Event 1推断状态和原始字节映射。
    state_bytes_table.reset_index().to_csv(
        OUTPUT_DIR / 'event1_state_bytes.csv', index=False
    )
    packet_state.rename('value').to_csv(
        OUTPUT_DIR / 'event1_packet_state.csv', header=True
    )
    catalogue_summary.rename('value').to_csv(
        OUTPUT_DIR / 'catalogue_summary.csv', header=True
    )

    # 3. 全目录竞争、冠军六分量和权重稳健性。
    final_rank.to_csv(
        OUTPUT_DIR / 'catalogue_candidates.csv', index_label='rank'
    )
    position_components.to_csv(
        OUTPUT_DIR / 'winner_position_components.csv', index=False
    )
    velocity_components.to_csv(
        OUTPUT_DIR / 'winner_velocity_components.csv', index=False
    )
    component_summary.rename('value').to_csv(
        OUTPUT_DIR / 'winner_component_summary.csv', header=True
    )
    sensitivity_table.to_csv(
        OUTPUT_DIR / 'velocity_weight_sensitivity.csv', index=False
    )
    sensitivity_top2.to_csv(
        OUTPUT_DIR / 'sensitivity_top2_details.csv', index=False
    )

    # 4. 站星几何、逐chirp CFO和保守经验误差。
    event_geometry.to_csv(OUTPUT_DIR / 'event_pair_geometry.csv')
    cfo_uncertainty.to_csv(OUTPUT_DIR / 'observed_cfo_uncertainty.csv')
    cfo_per_chirp.to_csv(
        OUTPUT_DIR / 'preamble_cfo_per_chirp.csv', index=False
    )
    cfo_difference_summary = pd.Series({
        'event1_cfo_hz': cfo_uncertainty.loc[
            'Event 1', 'preamble_median_cfo_hz'
        ],
        'event2_cfo_hz': cfo_uncertainty.loc[
            'Event 2', 'preamble_median_cfo_hz'
        ],
        'event2_minus_event1_cfo_hz': observed_delta_cfo_hz,
        'conservative_empirical_uncertainty_hz': observed_delta_sigma_hz,
        'observed_interval_lower_hz': (
            observed_delta_cfo_hz - observed_delta_sigma_hz
        ),
        'observed_interval_upper_hz': (
            observed_delta_cfo_hz + observed_delta_sigma_hz
        ),
        'uncertainty_definition': (
            'Per-event sqrt(sample_std_of_15_chirp_CFOs^2 + '
            'FFT_grid_sigma^2); event difference by quadrature.'
        ),
    })
    cfo_difference_summary.rename('value').to_csv(
        OUTPUT_DIR / 'differential_cfo_summary.csv', header=True
    )

    # 5. 事件前历史TLE质量表、全部有效曲线和逐时刻包络。
    pre_event_alignment.to_csv(
        OUTPUT_DIR / 'pre_event_tle_alignment.csv', index=False
    )
    tle_envelope = pd.DataFrame({
        'time_from_event1_s': envelope_times_s,
        'lower_hz': envelope_lower_hz,
        'upper_hz': envelope_upper_hz,
        'nominal_latest_pre_event_tle_hz': nominal_curve_hz,
    })
    tle_envelope.to_csv(
        OUTPUT_DIR / 'pre_event_tle_doppler_envelope.csv', index=False
    )
    curve_tables = []
    for curve_index, (satellite, curve_hz) in enumerate(
        zip(valid_pre_event_tles, pre_event_curves_hz)
    ):
        curve_tables.append(pd.DataFrame({
            'curve_index': curve_index,
            'tle_epoch_utc': satellite.epoch.utc_datetime().isoformat(),
            'time_from_event1_s': envelope_times_s,
            'relative_doppler_hz': curve_hz,
        }))
    pre_event_tle_curves = pd.concat(curve_tables, ignore_index=True)
    pre_event_tle_curves.to_csv(
        OUTPUT_DIR / 'pre_event_tle_doppler_curves.csv', index=False
    )
    tle_sensitivity_summary = pd.Series({
        'historical_records': len(historical_target_tles),
        'unique_dynamical_solutions': len(unique_historical_target_tles),
        'pre_event_solutions_checked': len(pre_event_tles),
        'valid_envelope_curves': len(valid_pre_event_tles),
        'nominal_latest_pre_event_epoch_utc': (
            valid_pre_event_tles[nominal_index].epoch.utc_datetime().isoformat()
        ),
        'event2_envelope_lower_hz': envelope_lower_hz[-1],
        'event2_envelope_upper_hz': envelope_upper_hz[-1],
        'observed_interval_fully_inside_envelope': bool(
            observed_delta_cfo_hz - observed_delta_sigma_hz
            >= envelope_lower_hz[-1]
            and observed_delta_cfo_hz + observed_delta_sigma_hz
            <= envelope_upper_hz[-1]
        ),
        'interpretation': (
            'Sensitivity range across quality-filtered pre-event TLEs; '
            'not a statistical confidence interval.'
        ),
    })
    tle_sensitivity_summary.rename('value').to_csv(
        OUTPUT_DIR / 'pre_event_tle_sensitivity_summary.csv', header=True
    )

    # 6. 正文图清单。六维分量图不列入正文，但其CSV仍在上面保存。
    selected_figures = pd.DataFrame([
        {
            'figure': 'Figure 1',
            'stem': 'figure10_1_all_satellites_w100',
            'purpose': 'Full-catalogue position-velocity error plane at W=100 s',
        },
        {
            'figure': 'Figure 2',
            'stem': 'figure10_2_weight_top2_error_plane',
            'purpose': 'Winner and runner-up sensitivity to velocity weight W',
        },
        {
            'figure': 'Figure 3',
            'stem': 'figure10_4b_pre_event_tle_sensitivity_envelope',
            'purpose': 'Pre-event TLE Doppler envelope and observed differential CFO',
        },
    ])
    selected_figures['png_exists'] = selected_figures['stem'].map(
        lambda stem: (OUTPUT_DIR / f'{stem}.png').is_file()
    )
    selected_figures['pdf_exists'] = selected_figures['stem'].map(
        lambda stem: (OUTPUT_DIR / f'{stem}.pdf').is_file()
    )
    selected_figures.to_csv(
        OUTPUT_DIR / 'selected_main_figures.csv', index=False
    )

    # 小型输入文件记录SHA-256；多GB原始电压只记录元数据，避免重复全盘读取。
    def file_sha256(path, block_bytes=1024 * 1024):
        digest = hashlib.sha256()
        with Path(path).open('rb') as handle:
            for block in iter(lambda: handle.read(block_bytes), b''):
                digest.update(block)
        return digest.hexdigest()

    manifest_rows = []
    small_inputs = [
        ('event1_summary', EVENT_DIRS['Event 1'] / 'summary.json'),
        ('event1_frame', EVENT_DIRS['Event 1'] / 'packet_000_frame.bin'),
        ('event2_summary', EVENT_DIRS['Event 2'] / 'summary.json'),
        ('event2_frame', EVENT_DIRS['Event 2'] / 'packet_000_frame.bin'),
        ('active_satellite_tle', TLE_PATH),
        ('starlink_32838_history_tle', HISTORICAL_TLE_PATH),
    ]
    for role, path in small_inputs:
        path = Path(path)
        manifest_rows.append({
            'role': role, 'path': str(path), 'size_bytes': path.stat().st_size,
            'sha256': file_sha256(path),
        })
    for label, summary in summaries.items():
        path = Path(summary['input_path'])
        manifest_rows.append({
            'role': f'{label.lower().replace(chr(32), chr(95))}_raw_voltage',
            'path': str(path), 'size_bytes': path.stat().st_size,
            'sha256': '',
        })
    input_manifest = pd.DataFrame(manifest_rows)
    input_manifest.to_csv(
        OUTPUT_DIR / 'reproduction_input_manifest.csv', index=False
    )

    def plain(value):
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, (datetime, Path)):
            return value.isoformat() if isinstance(value, datetime) else str(value)
        if isinstance(value, dict):
            return {str(key): plain(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [plain(item) for item in value]
        return value

    def series_dict(series):
        return {str(key): plain(value) for key, value in series.items()}

    result_summary = {
        'scope': (
            'Target identification for two decoded 137.055 MHz Starlink-VHF '
            'events; not flux-density or beam calibration.'
        ),
        'inputs': {
            'event1_raw': summaries['Event 1']['input_path'],
            'event2_raw': summaries['Event 2']['input_path'],
            'active_satellite_tle': str(TLE_PATH),
            'historical_target_tle': str(HISTORICAL_TLE_PATH),
            'manifest_csv': str(OUTPUT_DIR / 'reproduction_input_manifest.csv'),
        },
        'search_method': {
            'frame': 'ECEF/ITRS',
            'time_bound_s': TIME_BOUND_S,
            'velocity_weight_s': VELOCITY_WEIGHT_S,
            'objective': 'position_error_km^2 + (W_s * velocity_error_km_s)^2',
            'optimizer': 'scipy.optimize.minimize_scalar(method=bounded)',
            'full_exact_search': True,
            'catalogue_unique_norad_ids': len(satellites),
            'precisely_refined_candidates': len(final_rank),
        },
        'winner': {key: plain(winner[key]) for key in result_columns},
        'runner_up': {key: plain(runner_up[key]) for key in result_columns},
        'packet_pair': series_dict(packet_pair_summary),
        'component_summary': series_dict(component_summary),
        'legacy_single_tle_doppler_summary': series_dict(doppler_summary),
        'cfo_measurement': {
            'event1': series_dict(cfo_uncertainty.loc['Event 1']),
            'event2': series_dict(cfo_uncertainty.loc['Event 2']),
            'difference': series_dict(cfo_difference_summary),
            'fixed_timing_shift_samples': 0,
            'interpretation': (
                'Conservative chirp-repeatability uncertainty, not a formal '
                'confidence interval for the median.'
            ),
        },
        'pre_event_tle_sensitivity': series_dict(tle_sensitivity_summary),
        'selected_main_figures': plain(selected_figures.to_dict('records')),
        'conclusion': (
            'Event 1 inferred six-dimensional state uniquely strongly matches '
            'STARLINK-32838 in the full active-satellite catalogue. Event 2 is '
            'linked through packet continuity and differential Doppler; it is '
            'not an independent state-vector catalogue identification.'
        ),
        'cautions': [
            'Payload bytes 39-62 use an inferred ECEF int32/1e5 interpretation.',
            'Protocol-like ID 5927 is not treated as a NORAD catalogue number.',
            'Best TLE model time is not interpreted as the reception UTC.',
            'The historical-TLE envelope is a sensitivity range, not a confidence interval.',
        ],
    }
    summary_path = OUTPUT_DIR / 'target_identification_summary.json'
    summary_path.write_text(
        json.dumps(result_summary, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    print('Saved structured validation results to:', OUTPUT_DIR)
    display(selected_figures)
    display(tle_sensitivity_summary.to_frame('value'))
    display(cfo_difference_summary.to_frame('value'))
else:
    print('SAVE_RESULTS=False，没有写出文件。')


## 12. SpaceX-E SupGP独立精化验证

本节使用CelesTrak按申请提供的`SpaceX-E` SupGP历史文件，对第6节全活动卫星目录盲筛得到的STARLINK-32838进行**事后独立精化验证**。逻辑顺序不能倒置：

1. 原有`Active_0415.tle`全目录竞争负责回答“在不知道目标身份时，哪颗卫星最匹配Event 1推断状态”；
2. `sat000062827.csv`只包含已经选出的NORAD 62827，因此不能替代盲筛，也不能用于证明“其他卫星都更差”；
3. SupGP来自`SpaceX-E`运营商星历拟合，事件前最近历元距离payload GPS状态时刻仅约4.4 h，可检验更接近事件的轨道资料是否继续支持同一六维状态；
4. CSV中的`RMS`是SGP4对源星历的拟合残差，不是位置协方差、1σ误差或95%置信区间。

本节调用同目录的`supgp_validation.py`。脚本对事件日每条SupGP执行与第6节相同的ECEF/ITRS状态比较和$W=100$ s、$\pm180$ s优化，同时计算两事件差分多普勒及SupGP轨道演化。所有新结果使用`supgp_`前缀，不覆盖前面的盲筛结果。


In [ ]:
from datetime import datetime
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

# 本节可在重启kernel后独立运行，不需重做前面的全星表盲筛。
def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'project_config.py').is_file():
            return candidate
    raise FileNotFoundError('Run from this repository or one of its subdirectories')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_config import DATA_DIR, RAW_DATA_DIR, RESULTS_DIR, file_start_utc
ROOT = PROJECT_ROOT
OUTPUT_DIR = RESULTS_DIR / '3_satellite_identification'
SUPGP_PATH = DATA_DIR / 'orbital' / 'STARLINK-32838_SpaceX-E_SupGP.csv'
SUPGP_MODULE_PATH = PROJECT_ROOT / '3_satellite_identification' / 'supgp_validation.py'
STATION_CENTER_PATH = RESULTS_DIR / '2_waterfall' / 'e08_station_center_wgs84.csv'

assert SUPGP_PATH.is_file(), SUPGP_PATH
assert SUPGP_MODULE_PATH.is_file(), SUPGP_MODULE_PATH
assert STATION_CENTER_PATH.is_file(), STATION_CENTER_PATH

spec = spec_from_file_location('starlink_supgp_validation', SUPGP_MODULE_PATH)
supgp_module = module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(supgp_module)

supgp_summary = supgp_module.run_validation(
    SUPGP_PATH, OUTPUT_DIR, STATION_CENTER_PATH
)
supgp_state_utc = datetime.fromisoformat(
    supgp_summary['event1_packet_state']['gps_state_utc']
)
print('SupGP records:', supgp_summary['input']['records'])
print('SupGP source:', supgp_summary['input']['data_source'])
print('Epoch range:', supgp_summary['input']['epoch_start_utc'], 'to', supgp_summary['input']['epoch_stop_utc'])
print('Saved outputs under:', OUTPUT_DIR)


### 12.1 输入资料、时间关系和拟合质量

申请得到的文件应只包含STARLINK-32838（NORAD 62827），`DATA_SOURCE=SpaceX-E`。首先检查记录数、epoch范围和事件日前后的历元。事件状态时间继续使用第4节从payload解析的`gps_state_utc=2025-04-15 14:00:00.250 UTC`，不是文件acquisition metadata反推的接收时刻。

CelesTrak SupGP是把运营商提供的源星历拟合为可由SGP4传播的平均轨道根数。SupGP按其epoch向未来使用；事件前09:35:42 UTC记录传播到14:00 UTC约4.405 h，位于约8 h拟合/预报尺度内。`RMS`只用于评价该SGP4解对源星历的拟合质量，不能作为状态协方差使用。


In [ ]:
supgp_orbit = pd.read_csv(OUTPUT_DIR / 'supgp_orbit_evolution.csv')
supgp_orbit['epoch_utc'] = pd.to_datetime(
    supgp_orbit['epoch_utc'], utc=True, format='mixed'
)

event_day_supgp = supgp_orbit[
    supgp_orbit['epoch_utc'].dt.date == supgp_state_utc.date()
].copy()

display(event_day_supgp[[
    'epoch_utc', 'mean_motion_rev_day', 'mean_altitude_km',
    'mean_motion_dot_rev_day2', 'bstar', 'rms_fit_km', 'data_source',
]])


### 12.2 对Event 1推断六维状态的精化检验

下一表同时保留两种比较：

- `exact_*`：把每条SupGP直接传播到payload GPS状态时刻，不允许人为移动时间；
- `best_*`：为了与原全目录方法保持一致，再使用$W=100$ s对$\pm180$ s时间偏移做有界优化。



In [ ]:
supgp_state_match = pd.read_csv(OUTPUT_DIR / 'supgp_state_match.csv')
supgp_state_match['epoch_utc'] = pd.to_datetime(
    supgp_state_match['epoch_utc'], utc=True, format='mixed'
)
supgp_comparison = pd.read_csv(OUTPUT_DIR / 'supgp_vs_standard_tle.csv')

state_columns = [
    'epoch_utc', 'epoch_age_at_state_h', 'rms_fit_km',
    'exact_position_error_km', 'exact_velocity_error_m_s',
    'best_offset_s', 'best_position_error_km',
    'best_velocity_error_m_s', 'joint_score_km2',
]
display(supgp_state_match[state_columns])
display(supgp_comparison)

selected_supgp = supgp_summary['selected_pre_event_supgp']
print('Selected pre-event SupGP epoch:', selected_supgp['epoch_utc'])
print('Exact-time position error (km):', selected_supgp['exact_position_error_km'])
print('Exact-time velocity error (m/s):', selected_supgp['exact_velocity_error_m_s'])
print('Optimized time offset (s):', selected_supgp['best_offset_s'])
print('Optimized position error (km):', selected_supgp['best_position_error_km'])
print('Optimized velocity error (m/s):', selected_supgp['best_velocity_error_m_s'])


### 12.3 两事件差分多普勒复核

使用事件前最近的09:35 UTC SupGP，并沿用Event 1六维状态得到的约20 ms最佳时间校正，计算21CMA站心至卫星的距离、距离变化率和137.055 MHz几何多普勒。绝对CFO仍含发射机/接收机频率偏置，因此只比较Event 2减Event 1的约100 s差分量。




In [ ]:
supgp_geometry = pd.read_csv(
    OUTPUT_DIR / 'supgp_differential_doppler.csv', index_col=0
)
supgp_doppler = pd.Series(supgp_summary['differential_doppler'])

display(supgp_geometry)
display(supgp_doppler.to_frame('value'))


### 12.4 SupGP轨道演化与事件时的轨道抬升

`mean_altitude_km`由SupGP平均运动通过二体关系换算为平均半长轴相对于WGS84赤道半径的高度，只用于显示轨道能量随时间的演化，不是瞬时星下点高度。事件日前后平均运动先增加、随后明显减小，对应平均轨道高度在4月15日开始上升。

负`BSTAR`和较大的负`MEAN_MOTION_DOT`不应解释成真实“负大气阻力”；在这一阶段它们更可能是SGP4平均根数为拟合持续推力/轨道机动而使用的等效参数。4月16日以后拟合RMS升高，也提示单个SGP4解对快速机动源星历的近似变差。


In [ ]:
figure14_path = OUTPUT_DIR / 'figure14_1_supgp_orbit_evolution.png'
assert figure14_path.is_file()
display(Image(filename=str(figure14_path), width=1000))

orbit_day_summary = pd.Series(supgp_summary['event_day_orbit'])
display(orbit_day_summary.to_frame('value'))
